# CIFAKE Coursework Experiments
This notebook is a Colab driver for the shared code in `src/`. Use a GPU runtime, set `REPO_URL`, and run cells in order.

In [ ]:
REPO_URL = ""  # Set this to the Git URL for this repository.
assert REPO_URL, "Set REPO_URL before continuing"
!rm -rf /content/fake-image-detection
!git clone $REPO_URL /content/fake-image-detection
%cd /content/fake-image-detection
!pip -q install -r requirements.txt

## Download CIFAKE
Upload your Kaggle API token when prompted. The token is used only in this runtime.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select kaggle.json
!mkdir -p ~/.kaggle /content/data
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images -p /content/data --unzip
!mkdir -p /content/data/cifake
!cp -r /content/data/train /content/data/cifake/train
!cp -r /content/data/test /content/data/cifake/test
!sed -i 's#root: data/cifake#root: /content/data/cifake#' configs/cifake_*.yaml

## Audit and tests

In [ ]:
!python -m src.check_dataset --root /content/data/cifake
!python -m unittest discover -s tests -v

## Smoke tests
Create temporary one-epoch configurations and separate output directories so smoke tests cannot overwrite final evidence.

In [ ]:
import yaml
CONFIGS = [
    'configs/cifake_cnn.yaml',
    'configs/cifake_hybrid.yaml',
    'configs/cifake_hybrid_no_augmentation.yaml',
]
for config_path in CONFIGS:
    with open(config_path) as handle:
        smoke = yaml.safe_load(handle)
    name = config_path.split('/')[-1].removesuffix('.yaml')
    smoke['training']['epochs'] = 1
    smoke['training']['patience'] = 1
    smoke['training']['output_dir'] = f'runs/smoke_{name}'
    smoke_path = f'/tmp/smoke_{name}.yaml'
    with open(smoke_path, 'w') as handle:
        yaml.safe_dump(smoke, handle)
    !python -m src.train --config {smoke_path}

## Train final experiments

In [ ]:
for config_path in CONFIGS:
    !python -m src.train --config {config_path}

## Evaluate all final checkpoints

In [ ]:
RUNS = [
    ('configs/cifake_cnn.yaml', 'runs/cifake_cnn'),
    ('configs/cifake_hybrid.yaml', 'runs/cifake_hybrid'),
    ('configs/cifake_hybrid_no_augmentation.yaml', 'runs/cifake_hybrid_no_augmentation'),
]
for config, run in RUNS:
    !python -m src.evaluate --config {config} --checkpoint {run}/best_model.pt
!python -m src.compare runs/cifake_cnn runs/cifake_hybrid runs/cifake_hybrid_no_augmentation

## Preserve evidence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf '/content/drive/MyDrive/cifake_coursework_runs'
!cp -r runs '/content/drive/MyDrive/cifake_coursework_runs'